In [11]:
!pip install --upgrade ipywidgets
!jupyter nbextension enable --py widgetsnbextension


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 914.9/914.9 kB 32.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 79.9 MB/s  0:00:00
ERROR: Could not install packages due to an OSError: [Errno 30] Read-only file system: '/opt/conda/lib/python3.10/site-packages/widgetsnbextension'

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0/3 [widgetsnbextension]
/bin/bash: /home/chengjie.zheng001/.local/bin/jupyter: /usr/bin/python3: bad interpreter: No such file or directory


In [9]:
import sys
import os
sys.path.append(os.path.abspath(".."))  # 如果 notebooks 在项目根目录下的 notebooks/ 下

In [12]:
import os, sys
from pathlib import Path

def set_repo_root(marker_dir="time_moe"):
    """
    Find the nearest parent directory that contains `marker_dir/`,
    add it to sys.path, and chdir there.
    """
    p = Path.cwd().resolve()
    while p != p.parent:
        if (p / marker_dir).exists():
            if str(p) not in sys.path:
                sys.path.insert(0, str(p))
            os.chdir(str(p))
            return p
        p = p.parent
    raise RuntimeError(f"Cannot find repo root containing '{marker_dir}/'. "
                       f"Current cwd={Path.cwd().resolve()}")

repo_root = set_repo_root("time_moe")
print("✅ repo_root:", repo_root)

import time_moe
print("✅ time_moe imported from:", time_moe.__file__)


✅ repo_root: /beacon/data01/chengjie.zheng001/Projects/09PseudoTimeMoE/26KDD/WARP-MoE_26KDD
✅ time_moe imported from: /beacon/data01/chengjie.zheng001/Projects/09PseudoTimeMoE/26KDD/WARP-MoE_26KDD/time_moe/__init__.py


In [13]:
import torch
from torch.utils.data import DataLoader

from time_moe.datasets.mirage_cmapss_dataset import (
    MirageCmapssWindowDataset,
    EntityBatchSampler,
    mirage_collate_fn
)

# MIRAGE window hyperparams
L = 64
stride = 16

ds = MirageCmapssWindowDataset(
    hf_name="nominal-io/nasa-turbofan-degradation",
    split="train",
    context_length=L,
    stride=stride,
    normalize=True,
)

print("num windows:", len(ds))
print("num entities with windows:", len(ds.entity_to_indices))
print("example entity window count:", list(ds.entity_to_indices.items())[:1])

B = 8   # entities per batch
K = 4   # windows per entity
num_batches = 5

sampler = EntityBatchSampler(
    entity_to_indices=ds.entity_to_indices,
    batch_entities=B,
    windows_per_entity=K,
    num_batches=num_batches,
    seed=9899,
    shuffle_entities=True,
    replace_entities=False,
    replace_windows=False,
)

loader = DataLoader(ds, batch_sampler=sampler, collate_fn=mirage_collate_fn, num_workers=0)

batch = next(iter(loader))
for k, v in batch.items():
    print(k, v.shape, v.dtype)

# sanity: entity_ids should be grouped by entity within each K block only if sampler is implemented that way
# Here we simply reshape as [B, K] and check each row constant.
eid = batch["entity_ids"].view(B, K)
ts = batch["timestamps"].view(B, K)

print("entity_ids (B x K) first 3 rows:\n", eid[:3])
print("timestamps (B x K) first 3 rows:\n", ts[:3])

assert torch.all(eid == eid[:, :1]), "Each row in entity_ids should be constant (same entity)."
print("OK: entity-batched structure verified.")


num windows: 992
num entities with windows: 100
example entity window count: [(1, [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19])]
input_ids torch.Size([32, 64, 24]) torch.float32
labels torch.Size([32, 64, 24]) torch.float32
loss_masks torch.Size([32, 64]) torch.float32
entity_ids torch.Size([32]) torch.int64
timestamps torch.Size([32]) torch.float32
entity_ids (B x K) first 3 rows:
 tensor([[78, 78, 78, 78],
        [39, 39, 39, 39],
        [47, 47, 47, 47]])
timestamps (B x K) first 3 rows:
 tensor([[128.,  64., 160.,  80.],
        [112.,  64.,  96.,  80.],
        [ 96., 176.,  64., 160.]])
OK: entity-batched structure verified.
